**  1. Model Selection

Chosen Model: mistralai/Mistral-7B-Instruct-v0.2

A 7B parameter instruction-tuned LLM.

Efficient for instruction-following tasks.

Used in 4-bit quantized form with bitsandbytes to reduce memory usage and enable training on smaller GPUs.

2. Dataset Preparation
Dataset Used: dzunggg/legal-qa-v1

A legal Q&A dataset ideal for fine-tuning a model in the legal domain.

Steps Taken:

Prompt Engineering: Added a system instruction:

"You are a helpful legal assistant. Answer the user's legal questions clearly and accurately."

Formatting: Used Mistral's chat template (tokenizer.apply_chat_template) to format each example as a conversation between system, user, and assistant.

Cleaning: Removed Q: and A: prefixes from the dataset.

Splitting:

90% for training

10% for evaluation

3. Training Configuration
Libraries Used: transformers, peft, trl, datasets, evaluate, bert-score, bitsandbytes

PEFT Configuration (LoRA):

r=16, alpha=32, dropout=0.05

Targeted all Linear4bit layers excluding lm_head

Training Arguments:

Model trained using SFTTrainer from TRL.

Batch size per device: 4 (train), 1 (eval)

fp16 enabled

Optimizer: paged_adamw_32bit

Epochs: 1

Gradient Accumulation: 2

Learning Rate: 2e-4

4. Compute Used
Quantization: 4-bit (bnb_4bit) to save memory

Device: cuda:0 (GPU); fallback to CPU if unavailable (with warning)

Memory Cleanup: torch.cuda.empty_cache() and gc.collect() used post-batch to manage memory.
 5. Evaluation
Metrics Used:

F1 Score: Custom simple token-overlap-based F1

ROUGE-L: Via evaluate.load("rouge")

BERTScore (F1): Semantic similarity using bert-score

Process:

Evaluation before and after fine-tuning using the same held-out test set.

Batched generation using model.generate(...).text**

In [ ]:
# STEP 1: Install required libraries
!pip install -q transformers accelerate datasets peft bitsandbytes trl evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install evaluate

In [ ]:
pip install transformers datasets peft trl evaluate bert-score bitsandbytes tqdm huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.6 MB/s eta 0:00:00


In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=04a88749657b6ecc18c61c21ed4aea955c807bd614641d295536fa63230a092a
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [ ]:
!pip install --upgrade --no-cache-dir pandas datasets evaluate transformers peft trl rouge_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 160.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 149.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 114.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 238.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.2/484.2 kB 220.4 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.28.1
    Uninstalling huggingface-hub-0.28.1:
      Successfully uninstalled huggingface-hub-0.28.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.48.3
    Uninstalling transformers-4.48.3:
      Successfully uninstalled transformers-4.48.3
  Attempting uninstall: peft
    Found existing installation: peft 0.14.

In [ ]:
from evaluate import load  # Import the load function from the evaluate library
rouge = load("rouge")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from huggingface_hub import login
login()

In [ ]:
import os
import gc
import torch
import bitsandbytes as bnb

from tqdm import tqdm
from datasets import load_dataset
from evaluate import load
from sklearn.metrics import f1_score
from bert_score import score as bert_score
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, setup_chat_format, SFTConfig

In [ ]:
base_model = "google/flan-t5-xl" # Changed to a publicly available model

In [ ]:
dataset = load_dataset("dzunggg/legal-qa-v1", split="train")


legal_qa_full.csv:   0%|          | 0.00/6.21M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3742 [00:00<?, ? examples/s]

In [ ]:
dataset[0]

{'question': 'Q: I was wondering if a pain management office is acting illegally/did an illegal action.. I was discharged as a patient from a pain management office after them telling me that a previous pain management specialist I saw administered a steroid shot wrong and I told them in the portal that I spoke to lawyers for advice but no lawsuit/case was created. It was maybe 1-2 months after I was discharged that I no longer have access to my patient portal with them. Every time I try to login I enter my credentials, wait a few seconds, and then I get re-directed back to the original screen where I have various options to login. I know I can speak to the office directly and ask them about what specifically is going on, talk to other lawyers if this is a violation of my rights, etc. but I was just wondering if anyone on this site would know if this action is in fact illegal. ',
 'answer': "A:In Kentucky, your situation raises questions about patient rights and medical records access.

**Since the model mistralai/Mistral-7B-Instruct-v0.2 already includes a built-in chat template that uses the [INST]...[/INST] format, there is no need to define a custom template manually. The tokenizer automatically applies the correct formatting using tokenizer.apply_chat_template().
** **bold text**

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")

print(tokenizer.chat_template)

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content'] %}
    {%- set loop_messages = messages[1:] %}
{%- else %}
    {%- set loop_messages = messages %}
{%- endif %}

{{- bos_token }}
{%- for message in loop_messages %}
    {%- if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}
        {{- raise_exception('After the optional system message, conversation roles must alternate user/assistant/user/assistant/...') }}
    {%- endif %}
    {%- if message['role'] == 'user' %}
        {%- if loop.first and system_message is defined %}
            {{- ' [INST] ' + system_message + '\n\n' + message['content'] + ' [/INST]' }}
        {%- else %}
            {{- ' [INST] ' + message['content'] + ' [/INST]' }}
        {%- endif %}
    {%- elif message['role'] == 'assistant' %}
        {{- ' ' + message['content'] + eos_token}}
    {%- else %}
        {{- raise_exception('Only user and assistant roles are supported, with the exception of an initial opt

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset

# Disable Weights & Biases
os.environ["WANDB_DISABLED"] = "true"

# Base model: Mistral
base_model = "mistralai/Mistral-7B-Instruct-v0.2"
dataset_name = "dzunggg/legal-qa-v1"

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Determine device
device = "cuda:0" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("Warning: 4-bit quantization typically requires a GPU. Loading on CPU might not work or be efficient.")

# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map=device,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model)

# Load dataset
dataset = load_dataset(dataset_name, split="train")

# System prompt
instruction = "You are a helpful legal assistant. Answer the user's legal questions clearly and accurately."

# Clean Q: and A: prefixes
def remove_prefix(example):
    if example['question'].startswith('Q:'):
        example['question'] = example['question'][2:].strip()
    if example['answer'].startswith('A:'):
        example['answer'] = example['answer'][2:].strip()
    return example

dataset = dataset.map(remove_prefix, num_proc=4)

def format_chat_template(row):
    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": row["question"]},
        {"role": "assistant", "content": row["answer"]}
    ]
    row["text"] = tokenizer.apply_chat_template(
    messages,
    tokenize=False
)

    return row

dataset = dataset.map(format_chat_template, num_proc=4)

# Split dataset
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

data%20UTF-8.csv:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5560 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/5560 [00:00<?, ? examples/s]

In [ ]:
def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:
        lora_module_names.remove('lm_head')
    return list(lora_module_names)

modules = find_all_linear_names(model)


In [ ]:
if tokenizer.chat_template is None:
    model, tokenizer = setup_chat_format(model, tokenizer)
else:
    print("Chat template already exists. Skipping setup_chat_format.")

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=modules
)

model = get_peft_model(model, peft_config)


Chat template already exists. Skipping setup_chat_format.


In [ ]:
# Before calling tokenizer in evaluate_model function:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def evaluate_model(model, tokenizer, dataset, max_new_tokens=60, batch_size=16):
    model.eval()
    results, f1_scores, rouge_preds, rouge_labels = [], [], [], []
    for i in tqdm(range(0, len(dataset), batch_size)):
        batch = dataset[i:i+batch_size]
        tokenizer.padding_side = "left"
        # Adjust model_max_length for tokenization
        tokenizer.model_max_length = 512 # Reduced to 512 to accommodate long text
        inputs = tokenizer(
            batch["text"], return_tensors="pt", padding=True, truncation=True,
            max_length=tokenizer.model_max_length # Using the reduced model_max_length
        ).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=max_new_tokens,
                do_sample=False,
            )
        decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        decoded_labels = batch["Answer"]
        for pred, label in zip(decoded_preds, decoded_labels):
            pred_clean = clean_prediction(pred)
            label_clean = label.strip()
            f1 = simple_f1(pred_clean, label_clean)
            f1_scores.append(f1)
            rouge_preds.append(pred_clean)
            rouge_labels.append(label_clean)
            results.append({"prediction": pred_clean, "reference": label_clean, "f1": f1})
        del inputs, outputs
        torch.cuda.empty_cache()
        gc.collect()
    avg_f1 = sum(f1_scores) / len(f1_scores)
    rouge_result = rouge.compute(predictions=rouge_preds, references=rouge_labels)
    rougeL = rouge_result["rougeL"]
    P, R, F1 = bert_score(rouge_preds, rouge_labels, lang="en", device="cpu", verbose=False)
    avg_bert_f1 = F1.mean().item()
    print(f"\n Average F1 Score:     {avg_f1:.4f}")
    print(f" ROUGE-L Score:        {rougeL:.4f}")
    print(f" BERTScore (F1):       {avg_bert_f1:.4f}")
    return {
        "f1": avg_f1,
        "rougeL": rougeL,
        "bertscore_f1": avg_bert_f1,
        "results": results
    }

In [ ]:

# Evaluate the base model
base_metrics = evaluate_model(model, tokenizer, eval_dataset, max_new_tokens=60)

# Training setup for fine-tuning
tokenizer.truncation_side = "left"
tokenizer.model_max_length = 1024

training_arguments = SFTConfig(
    output_dir="outputs",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    num_train_epochs=1,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    group_by_length=True,
    report_to="none",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    args=training_arguments
)

trainer.train()

  0%|          | 0/35 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
/usr/local/lib/python3.11/dist-packages/bitsandbytes/nn/modules.py:451: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(
  3%|▎         | 1/35 [00:41<23:40, 41.77s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
  6%|▌         | 2/35 [01:21<22:17, 40.53s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
A decoder-only architecture is being used, but right-padding was dete

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Tokenizing eval dataset:   0%|          | 0/556 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/556 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
500,0.670600


TrainOutput(global_step=625, training_loss=0.6485595947265625, metrics={'train_runtime': 1666.9182, 'train_samples_per_second': 3.002, 'train_steps_per_second': 0.375, 'total_flos': 1.5238306266710016e+16, 'train_loss': 0.6485595947265625})

In [ ]:
tokenizer.truncation_side = "left"
tokenizer.model_max_length = 1024

training_arguments = SFTConfig(
    output_dir="outputs",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    num_train_epochs=1,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    group_by_length=True,
    report_to="none",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    args=training_arguments
)

trainer.train()


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
500,0.436800


TrainOutput(global_step=625, training_loss=0.45202664794921876, metrics={'train_runtime': 1679.125, 'train_samples_per_second': 2.98, 'train_steps_per_second': 0.372, 'total_flos': 1.5238306266710016e+16, 'train_loss': 0.45202664794921876})

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
tokenizer.model_max_length = 512

fine_tuned_metrics = evaluate_model(model, tokenizer, eval_dataset)

print("\n====================== Evaluation Results ======================")
print(">>> Base Model:")
print(f"- Average F1 Score     : {base_metrics['f1']:.4f}")
print(f"- ROUGE-L Score        : {base_metrics['rougeL']:.4f}")
print(f"- BERTScore (F1)       : {base_metrics['bertscore_f1']:.4f}")

print("\n>>> Fine-Tuned Model:")
print(f"- Average F1 Score     : {fine_tuned_metrics['f1']:.4f}")
print(f"- ROUGE-L Score        : {fine_tuned_metrics['rougeL']:.4f}")
print(f"- BERTScore (F1)       : {fine_tuned_metrics['bertscore_f1']:.4f}")
print("===============================================================")

  0%|          | 0/35 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
  3%|▎         | 1/35 [00:17<10:11, 17.99s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
  6%|▌         | 2/35 [00:44<12:47, 23.25s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
  9%|▊         | 3/35 [00:59<10:23, 19.47s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
A decoder-only architecture is be


 Average F1 Score:     0.9816
 ROUGE-L Score:        0.9775
 BERTScore (F1):       0.9926

====================== Evaluation Results ======================
>>> Base Model:
- Average F1 Score     : 0.6041
- ROUGE-L Score        : 0.5303
- BERTScore (F1)       : 0.9346

>>> Fine-Tuned Model:
- Average F1 Score     : 0.9816
- ROUGE-L Score        : 0.9775
- BERTScore (F1)       : 0.9926
